In [2]:
import pandas as pd
import os

# Definimos la ruta a la carpeta donde están los Excels
ruta_raw = "../data/raw/"

# Listar todos los archivos que hay dentro
archivos_excel = [archivo for archivo in os.listdir(ruta_raw) if archivo.endswith('.xlsx')]

print(f"He encontrado {len(archivos_excel)} archivos Excel en la carpeta raw:")
for archivo in archivos_excel[:5]: 
    print(f"- {archivo}")

He encontrado 22 archivos Excel en la carpeta raw:
- LISTADO SEMANAL TICKETS PENDIENTES 03-07-2026.xlsx
- LISTADO SEMANAL TICKETS PENDIENTES 04-09-2026.xlsx
- LISTADO SEMANAL TICKETS PENDIENTES 05-06-2026.xlsx
- LISTADO SEMANAL TICKETS PENDIENTES 07-08-2026.xlsx
- LISTADO SEMANAL TICKETS PENDIENTES 08-05-2026.xlsx


In [5]:
import pandas as pd
import os

ruta_raw = "../data/raw/"
archivos_excel = [archivo for archivo in os.listdir(ruta_raw) if archivo.endswith('.xlsx')]

# Que Python busque el archivo que contenga la palabra "Tareas"
archivo_maestro = [f for f in archivos_excel if "Tareas" in f][0]
ruta_maestro = os.path.join(ruta_raw, archivo_maestro)

# Que Python coja el primer snapshot de la lista
archivo_snapshot = [f for f in archivos_excel if "LISTADO" in f][0]
ruta_snapshot = os.path.join(ruta_raw, archivo_snapshot)

print(f"--- Cargando Excel Maestro: {archivo_maestro} ---")
df_maestro = pd.read_excel(ruta_maestro, sheet_name=0) # Lee la primera pestaña
print(f"Filas: {len(df_maestro)}. Columnas del maestro:")
print(df_maestro.columns.tolist())

print(f"\n--- Cargando Snapshot Semanal: {archivo_snapshot} ---")
df_snapshot = pd.read_excel(ruta_snapshot, sheet_name=0)
print(f"Filas: {len(df_snapshot)}. Columnas del snapshot:")
print(df_snapshot.columns.tolist())

--- Cargando Excel Maestro: Tareas mmto prl seguridad ssgg 2026 (1).xlsx ---
Filas: 46. Columnas del maestro:
['EXPLICACIÓN TAREA SEGUIMIENTO CBRE']

--- Cargando Snapshot Semanal: LISTADO SEMANAL TICKETS PENDIENTES 03-07-2026.xlsx ---
Filas: 94. Columnas del snapshot:
['Work Order', 'PDS', 'Description', 'Comentarios', 'Location Description', 'Status', 'Target Start', 'Priority', 'Work Type', 'Reported By', 'Vendor / Tech', 'RITM']


In [7]:
import pandas as pd
import os

ruta_raw = "../data/raw/"
archivos_excel = [archivo for archivo in os.listdir(ruta_raw) if archivo.endswith('.xlsx')]

# 1. CARGAR EL MAESTRO (Pestaña 'Facilities')
archivo_maestro = [f for f in archivos_excel if "Tareas" in f][0]
ruta_maestro = os.path.join(ruta_raw, archivo_maestro)

print(f"--- Cargando Excel Maestro: {archivo_maestro} ---")

df_maestro = pd.read_excel(ruta_maestro, sheet_name='Facilities') 
print(f"Filas: {len(df_maestro)}. Columnas del maestro:")
print(df_maestro.columns.tolist())

# 2. CARGAR EL SNAPSHOT
archivo_snapshot = [f for f in archivos_excel if "LISTADO" in f][0]
ruta_snapshot = os.path.join(ruta_raw, archivo_snapshot)

print(f"\n--- Cargando Snapshot Semanal: {archivo_snapshot} ---")
df_snapshot = pd.read_excel(ruta_snapshot, sheet_name=0)
print(f"Filas: {len(df_snapshot)}. Columnas del snapshot:")
print(df_snapshot.columns.tolist())

--- Cargando Excel Maestro: Tareas mmto prl seguridad ssgg 2026 (1).xlsx ---
Filas: 1649. Columnas del maestro:
['s', 'OFICINA', 'AREA / ZONA\n\n (Espacio donde se detecta la incidencia)', 'Incidencia \n\n(ILUMINACION\nTEMPERATURA\nHGUMEDAD,,,)', 'ORIGEN INCIDENCIA ', 'MEDIDA PREVETIVA O TAREA A  REALIZAR\n  \n(A COMPLETAR POR PRL / SEGURIDAD / FACILITIES)', 'FECHA DE DETECCION DE LA INCIDENCIA', 'FECHA PROBABLE RESOLUCION', 'FECHA REALIZACION ', 'Fecha ticket', 'Ticket CBRE ', 'INFORME DONDE APARECE\n\n (auditoria, evcaluacion riesgos, simulaccro, parte de mantenimiento)', 'Prioridad', 'Comentario PRL / Facilities / Seguridad ', 'Comentarios Oficinas', 'Estado', 'Necesita evidencia?', 'Foto evidencia', 'Unnamed: 18', 'Unnamed: 19', 'RESPONSABLE IMPLANTACION', 'RESPONSABLE SEGUIMIENTO', 'ESTADO (ABIERTO / PENDIENTE / CERRADO)', 'Unnamed: 23', 'Unnamed: 24']

--- Cargando Snapshot Semanal: LISTADO SEMANAL TICKETS PENDIENTES 03-07-2026.xlsx ---
Filas: 94. Columnas del snapshot:
['Work Or

In [8]:
# 1. Diccionario para renombrar y seleccionar solo las columnas vitales
mapeo_columnas = {
    'Ticket CBRE ': 'ticket_id',
    'OFICINA': 'oficina',
    'Incidencia \n\n(ILUMINACION\nTEMPERATURA\nHGUMEDAD,,,)': 'descripcion',
    'Fecha ticket': 'fecha_creacion',
    'Estado': 'estado_maestro'
}

# 2. Filtro para quedarme solo con esas columnas y las renombramos
df_maestro_limpio = df_maestro[list(mapeo_columnas.keys())].rename(columns=mapeo_columnas)

# 3. Limpieza ajustada al negocio: 
# Se quita la fila SOLO si le falta tanto el ticket_id como la descripcion
df_maestro_limpio = df_maestro_limpio.dropna(subset=['ticket_id', 'descripcion'], how='all')

# 4. Homogeneizamos los datos
# Se rellenan los tickets vacíos con un texto estándar y se limpian los espacios
df_maestro_limpio['ticket_id'] = df_maestro_limpio['ticket_id'].fillna('SIN_TICKET').astype(str).str.strip()
df_maestro_limpio['descripcion'] = df_maestro_limpio['descripcion'].astype(str).str.strip()

print(f"Tras la limpieza ajustada, pasamos de 1649 filas a {len(df_maestro_limpio)} tickets reales.")
print("\nAsí luce ahora nuestra tabla maestra (primeras 5 filas):")
print(df_maestro_limpio.head(5))

Tras la limpieza ajustada, pasamos de 1649 filas a 1647 tickets reales.

Así luce ahora nuestra tabla maestra (primeras 5 filas):
               ticket_id   oficina  \
0             SIN_TICKET    Palma    
1              SR5437356   Viriato   
2  SR5469592 - SR5599667  Diagonal   
3              SR5332293   Acacias   
4                 687821   Granada   

                                         descripcion       fecha_creacion  \
0  Toda la luminaria exterior y Focos sobre cajer...  1905-07-10 00:00:00   
1  Reparación humedad/ gotera techo primera plant...  2021-11-03 00:00:00   
2                     Cristales rotos dos ascensores  2021-11-14 00:00:00   
3  Cambio equipo de luces Acacias: Techo y luces ...  2021-12-12 00:00:00   
4  Aún no se ha reparado rótulos de fachada y luz...  2022-01-07 07:30:00   

  estado_maestro  
0        Cerrado  
1        Cerrado  
2        Cerrado  
3        Cerrado  
4        Cerrado  


In [9]:
import re

# Se buscan solo los archivos que sean reportes semanales
archivos_snapshots = [f for f in archivos_excel if "LISTADO SEMANAL" in f]
lista_dfs = []

print(f"Procesando {len(archivos_snapshots)} reportes semanales...")

for archivo in archivos_snapshots:
    ruta = os.path.join(ruta_raw, archivo)
    df_temp = pd.read_excel(ruta, sheet_name=0)
    
    
    match = re.search(r'(\d{2}-\d{2}-\d{4})', archivo)
    if match:
        fecha_str = match.group(1)
        df_temp['fecha_reporte'] = pd.to_datetime(fecha_str, format='%d-%m-%Y')
    else:
        df_temp['fecha_reporte'] = pd.NaT 
    
    # Selección de columnas críticas para vigilar el SLA
    cols_utiles = ['Work Order', 'Description', 'Status', 'Target Start']
    cols_presentes = [c for c in cols_utiles if c in df_temp.columns]
    
    # Se guardan estas columnas más la fecha del reporte que acabamos de crear
    lista_dfs.append(df_temp[cols_presentes + ['fecha_reporte']])

# Se concatenan los 22 Excels en una sola base de datos temporal
df_historico_cbre = pd.concat(lista_dfs, ignore_index=True)

# Limpieza de ID
df_historico_cbre['Work Order'] = df_historico_cbre['Work Order'].astype(str).str.strip()

print(f"¡Fusión completada! Hemos creado un registro histórico con {len(df_historico_cbre)} observaciones de tickets.")
print("Muestra del historial (primeras 3 filas):")
print(df_historico_cbre.head(3))

Procesando 21 reportes semanales...
¡Fusión completada! Hemos creado un registro histórico con 1332 observaciones de tickets.
Muestra del historial (primeras 3 filas):
  Work Order                                        Description    Status  \
0    3870466  REPARACION LUZ ROTULO OFICINA - Reparación de ...  DISPATCH   
1    4454367  SUMINISTRO Y SUSTITUCION DE EMERGENCIAS - CORR...  DISPATCH   
2    4993411  Tenemos problemas eléctricos en oficina A Coru...  DISPATCH   

  Target Start fecha_reporte  
0   2025-10-13    2026-07-03  
1   2026-02-06    2026-07-03  
2   2026-05-21    2026-07-03  


In [10]:
import numpy as np

# Agrupar historial por ticket
df_agrupado = df_historico_cbre.groupby('Work Order').agg(
    fecha_inicio=('Target Start', 'min'),
    ultimo_reporte=('fecha_reporte', 'max')
).reset_index()

# Días vivo en el sistema
df_agrupado['dias_abierto'] = (df_agrupado['ultimo_reporte'] - df_agrupado['fecha_inicio']).dt.days

# Detectar fecha de último snapshot general
fecha_tope = df_agrupado['ultimo_reporte'].max()

def asignar_target(row):
    if row['dias_abierto'] > 21:
        return 1
    elif (fecha_tope - row['fecha_inicio']).days <= 21:
        # Ventana ciega: no ha pasado suficiente tiempo para asegurar clase 0
        return np.nan
    else:
        return 0

# Aplicar lógica y limpiar censurados
df_agrupado['target_sla'] = df_agrupado.apply(asignar_target, axis=1)
df_target = df_agrupado.dropna(subset=['target_sla']).copy()
df_target['target_sla'] = df_target['target_sla'].astype(int)

print(f"Total tickets únicos en snapshots: {len(df_agrupado)}")
print(f"Tickets válidos para ML (sin ventana ciega): {len(df_target)}")
print("\nDistribución del Target (0 = En tiempo, 1 = Retraso):")
print(df_target['target_sla'].value_counts())

Total tickets únicos en snapshots: 104
Tickets válidos para ML (sin ventana ciega): 104

Distribución del Target (0 = En tiempo, 1 = Retraso):
target_sla
1    79
0    25
Name: count, dtype: int64


In [11]:
# Cruzar tabla maestra con las etiquetas calculadas en los snapshots
df_cruce = pd.merge(df_maestro_limpio, df_target[['Work Order', 'target_sla']], 
                    left_on='ticket_id', right_on='Work Order', how='left')

# Rescatar tickets rápidos (Clase 0 implícita)
# Si está Cerrado en el maestro y no tiene etiqueta de retraso de los snapshots, cumplió el SLA
condicion_rapidos = (df_cruce['estado_maestro'] == 'Cerrado') & (df_cruce['target_sla'].isna())
df_cruce.loc[condicion_rapidos, 'target_sla'] = 0

# Descartar casos dudosos (Abiertos recientemente que aún no han superado 21 días)
df_final = df_cruce.dropna(subset=['target_sla']).copy()
df_final['target_sla'] = df_final['target_sla'].astype(int)

# Eliminar duplicados por si un mismo ID aparece dos veces en el maestro
df_final = df_final.drop_duplicates(subset=['ticket_id'])

print(f"Base de datos lista para Machine Learning: {len(df_final)} tickets.")
print("\nDistribución real del Target (0 = En tiempo, 1 = Retraso):")
print(df_final['target_sla'].value_counts())

Base de datos lista para Machine Learning: 1411 tickets.

Distribución real del Target (0 = En tiempo, 1 = Retraso):
target_sla
0    1340
1      71
Name: count, dtype: int64


In [12]:
# Guardar dataset
ruta_procesado = "../data/processed/tickets_etiquetados.csv"
df_final.to_csv(ruta_procesado, index=False, encoding='utf-8')

print(f"Dataset final guardado correctamente en: {ruta_procesado}")

Dataset final guardado correctamente en: ../data/processed/tickets_etiquetados.csv
